# Heart Failure Risk Prediction - Exploratory Data Analysis
## Objectif

Ce notebook analyse le dataset Heart Failure Clinical Records afin de :
- Comprendre la structure des données
- Identifier les problèmes (valeurs manquantes, outliers, déséquilibre)
- Préparer les données pour le modèle ML

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_processing import optimize_memory

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

ModuleNotFoundError: No module named 'pandas'

In [1]:
df = pd.read_csv('data/heart_failure_clinical_records.csv')
print("Shape:", df.shape)
df.head()


NameError: name 'pd' is not defined

In [ ]:
df.info()
df.describe()

### Observations
- 299 patients, 13 features
- Target : DEATH_EVENT (0 = survécu, 1 = décédé)
- Types : int64 et float64

In [ ]:

# C VALEURS MANQUANTES & OUTLIERS


# Valeurs manquantes
print("=== VALEURS MANQUANTES ===")
missing = df.isnull().sum()
print(missing)
print(f"\nTotal : {missing.sum()} valeurs manquantes")

# Outliers via méthode IQR
print("\n=== OUTLIERS (méthode IQR) ===")
Q1 = df.quantile(0.25)
Q3 = df.quantile(0.75)
IQR = Q3 - Q1
outliers_count = ((df < (Q1 - 1.5 * IQR)) | (df > (Q3 + 1.5 * IQR))).sum()
print("Outliers par colonne :\n", outliers_count)


### Observations
- ✅ Aucune valeur manquante → pas d'imputation nécessaire
- ⚠️ Outliers présents dans : creatinine_phosphokinase, serum_creatinine, platelets
- ✅ On les **conserve** car ce sont des valeurs médicalement réelles
- ✅ Les modèles ensemblistes (Random Forest, XGBoost) sont robustes aux outliers

# DÉSÉQUILIBRE DES CLASSES


In [ ]:
print("=== RÉPARTITION DES CLASSES ===")
print(df['DEATH_EVENT'].value_counts())
print("\nPourcentages :")
print(df['DEATH_EVENT'].value_counts(normalize=True) * 100)

plt.figure(figsize=(6,4))
sns.countplot(x='DEATH_EVENT', data=df, palette='viridis')
plt.title('Déséquilibre : Survie (0) vs Décès (1)')
plt.xticks([0, 1], ['Survécu (0)', 'Décédé (1)'])
plt.ylabel('Nombre de patients')
plt.show()

### Résultat
✅ Aucune valeur manquante détectée dans ce dataset.
Aucune action requise (pas d'imputation nécessaire).

In [ ]:
class_counts = df['DEATH_EVENT'].value_counts()
print(class_counts)
print(f"\nPourcentage survivants : {class_counts[0]/len(df)*100:.1f}%")
print(f"Pourcentage décédés   : {class_counts[1]/len(df)*100:.1f}%")

# Visualisation
plt.figure(figsize=(6,4))
sns.countplot(x='DEATH_EVENT', data=df, palette='Set2')
plt.title('Distribution de la variable cible')
plt.xticks([0,1], ['Survécu (0)', 'Décédé (1)'])
plt.ylabel('Nombre de patients')
plt.show()

### Observations
- ~68% survivants (classe 0) → 203 patients
- ~32% décédés (classe 1) → 96 patients
- ⚠️ Dataset déséquilibré → nécessite traitement

### Stratégie choisie : SMOTE
- On génère des exemples synthétiques de la classe minoritaire
- Préféré à l'undersampling car le dataset est petit (299 lignes seulement)

#  VISUALISATION OUTLIERS

In [ ]:
numerical_cols = ['age', 'creatinine_phosphokinase', 'ejection_fraction',
                  'platelets', 'serum_creatinine', 'serum_sodium', 'time']

# Boxplots AVANT traitement
plt.figure(figsize=(15, 5))
for i, col in enumerate(numerical_cols):
    plt.subplot(2, 4, i+1)
    sns.boxplot(y=df[col], color='skyblue')
    plt.title(col)
plt.suptitle('Outliers AVANT traitement', fontsize=13)
plt.tight_layout()
plt.show()

### Décision finale sur les outliers
- La fonction `handle_outliers()` est disponible dans `src/data_processing.py`
- Pour ce projet on **conserve les outliers** sans clipping car :
  - Les valeurs sont médicalement plausibles
  - Random Forest et XGBoost sont naturellement robustes
  - Clipper pourrait faire perdre de l'information clinique importante


In [ ]:
#Déséquilibre des classes
 class_counts = df['DEATH_EVENT'].value_counts()
print(class_counts)
print(f"\nPourcentage survivants : {class_counts[0]/len(df)*100:.1f}%")
print(f"Pourcentage décédés   : {class_counts[1]/len(df)*100:.1f}%")

# Visualisation
plt.figure(figsize=(6,4))
sns.countplot(x='DEATH_EVENT', data=df, palette='Set2')
plt.title('Distribution de la variable cible')
plt.xticks([0,1], ['Survécu (0)', 'Décédé (1)'])
plt.ylabel('Nombre de patients')
plt.show()

### Résultat
- ~68% survivants (classe 0) → 203 patients
- ~32% décédés (classe 1) → 96 patients
⚠️ Dataset déséquilibré !

### Stratégie choisie : SMOTE (oversampling)
SMOTE génère des exemples synthétiques de la classe minoritaire.
Choix justifié car :
- On ne veut pas perdre de données (undersampling réduit trop un petit dataset de 299 lignes)
- Le class_weight='balanced' seul est moins performant sur des modèles ensemblistes

corrélation 


In [ ]:


# Matrice de corrélation complète
plt.figure(figsize=(12, 8))
sns.heatmap(
    df.corr(),
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)
plt.title("Matrice de Corrélation - Heart Failure Dataset")
plt.tight_layout()
plt.show()

# Corrélation avec la variable cible uniquement
corr_target = df.corr()["DEATH_EVENT"].sort_values(ascending=False)

print("Corrélation avec DEATH_EVENT :")
print(corr_target)

# Graphique bar
corr_target.drop("DEATH_EVENT").plot(kind="bar", figsize=(10, 5), color='steelblue')
plt.title("Corrélation des features avec DEATH_EVENT")
plt.ylabel("Coefficient de corrélation")
plt.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
plt.tight_layout()
plt.show()


SyntaxError: invalid character '→' (U+2192) (3304508429.py, line 27)

: 

### Observations
- `time` : corrélation négative forte (-0.53) → plus le suivi est long, moins le risque de décès
- `serum_creatinine` : corrélation positive (0.29) → taux élevé = risque accru
- `ejection_fraction` : corrélation négative (-0.27) → fraction faible = risque accru
- Pas de corrélation très forte entre features → pas de multicolinéarité critique

### Stratégie
✅ Toutes les features sont conservées — SHAP confirmera les plus importantes après entraînement

Optimisation mémoire 


In [ ]:

from src.data_processing import optimize_memory

print("=== AVANT optimisation ===")
print(df.dtypes)
mem_before = df.memory_usage(deep=True).sum() / 1024
print(f"Mémoire : {mem_before:.2f} KB")

df_optimized = optimize_memory(df)

print("\n=== APRÈS optimisation ===")
print(df_optimized.dtypes)
mem_after = df_optimized.memory_usage(deep=True).sum() / 1024
print(f"Mémoire : {mem_after:.2f} KB")
print(f"Réduction : {((mem_before - mem_after)/mem_before*100):.1f}%")

### Résultat
- float64 → float32 : réduit de moitié la mémoire des colonnes numériques
- int64 → int32 : idem pour les colonnes entières
- Gain typique : ~50% de réduction mémoire


SMOTE


In [ ]:
from imblearn.over_sampling import SMOTE

X = df.drop('DEATH_EVENT', axis=1)
y = df['DEATH_EVENT']

print("=== AVANT SMOTE ===")
print(y.value_counts())

smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

print("\n=== APRÈS SMOTE ===")
print(pd.Series(y_resampled).value_counts())

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
pd.Series(y).value_counts().plot(kind='bar', ax=axes[0], 
    color=['steelblue','salmon'], title='Avant SMOTE')
pd.Series(y_resampled).value_counts().plot(kind='bar', ax=axes[1], 
    color=['steelblue','salmon'], title='Après SMOTE')
plt.tight_layout()
plt.show()

### Résultat
- Avant : 203 survivants / 96 décédés (68% / 32%)
- Après SMOTE : classes parfaitement équilibrées (50% / 50%)
- Méthode choisie : SMOTE car le dataset est petit (299 lignes)
  et l'undersampling ferait perdre trop de données

# Conclusion


## Résumé des décisions

| Problème | Constat | Solution |
|----------|---------|----------|
| Valeurs manquantes | Aucune | Rien à faire |
| Outliers | Présents mais médicaux | Conservés |
| Déséquilibre classes | 68% / 32% | SMOTE appliqué |
| Corrélations | Faibles entre features | Toutes conservées |
| Mémoire | Optimisable | optimize_memory() appliqué |

## Features les plus prometteuses (à confirmer avec SHAP)
1. `time` — durée de suivi
2. `serum_creatinine` — taux de créatinine
3. `ejection_fraction` — fraction d'éjection cardiaque